> **PRE-RUN COPY.** The outputs below were produced on 2026-10-13 on the team's own laptop (Windows, CPU only, 12 threads) by running every cell of this notebook with `MODE = "quick"` (XGBoost and ANN were really retrained). They are saved so the notebook can be shown without running it. To regenerate them on Colab or Kaggle, use **Run all**; neural-network numbers may differ by about 0.01 m.

# TGT-M: the full modeling pipeline (Review 2)
**Team 7 · SRM IST · 21CSC314P Big Data Essentials.** *TGT-M: A Temporal Gated Transformer Model for Seasonal Groundwater Stress Assessment and Cross-Region Generalization Across India Using Multimodal Data.*

This notebook shows **how the whole modeling study is organised and run**, retrains a small part of it live, and prints the result tables.

| Part | What it does |
|---|---|
| 1 | Finds the project files |
| 2 | Checks the data (253,828 well-rounds, 84 columns, leakage-safe model inputs) |
| 3 | **Code map**: every script in `modeling/`, what it does, and the order they run in |
| 4 | Runs the models: **`quick`** (XGBoost + ANN, 10-20 min), **`full`** (everything, hours) or **`view`** (nothing) |
| 5 | The result tables: main comparison, rainfall effect, unseen states, the two pre-registered tests |

### Pick a mode (`MODE` in Part 1)
* `view` — trains nothing; prints the stored results.
* `quick` — retrains **XGBoost and the ANN** (one seed) and compares with the published numbers. Writes to `results_quick/`; nothing else is touched.
* `full` — reproduces everything (7 models × 2 variants × 3 seeds, unseen-state tests, pre-registered tests, tables). About 10 CPU-hours: split it with `STEPS` and use *Save & Run All* on Kaggle.

### Protocol (identical for every model)
Train 2000-2015, choose settings on 2016-2017, test on 2018-2022 (40,630 rows). Model A = no rainfall, Model B = with rainfall. Target = capped anomaly z of a **hidden** historical reading; stress class from the predicted z (1 / 1.5 / 2). 3 seeds. The test split is never used for any choice.


## Part 1 · Settings and project files

In [1]:
MODE = "quick"          # "view" | "quick" | "full"
STEPS = "chrono,geo,robust,evaluate"   # full mode only; on Kaggle (12 h limit) run them in separate sessions

import os, sys, glob, zipfile, subprocess, time, shutil
from pathlib import Path
BUNDLE_ZIP = None      # e.g. "/content/bde_bundle_core.zip" if auto-detect fails
GITHUB_URL = "https://github.com/dhruvvvgg/bde-team7.git"   # only works after the project was pushed

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = (not ON_KAGGLE) and Path("/content").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB else Path.cwd()

def looks_ok(p):
    p = Path(p); return (p / "modeling" / "data.py").exists() and (p / "processed" / "gw_features_by_state").exists()

def writable(c):
    """Kaggle mounts datasets read-only; copy the project to the writable working folder so quick/full mode can write results."""
    c = Path(c)
    if str(c).startswith("/kaggle/input"):
        dest = WORK / "bde-team7"
        if not looks_ok(dest):
            print("copying the project from the read-only dataset to", dest, "...")
            shutil.copytree(c, dest, dirs_exist_ok=True)
        return dest
    return c

def find_root():
    cands = [os.environ.get("BDE_ROOT"), Path.cwd(), Path.cwd().parent, WORK / "bde-team7"]
    cands += [Path(p).parent.parent for p in glob.glob("/kaggle/input/**/modeling/data.py", recursive=True)]
    for c in cands:
        if c and looks_ok(c): return writable(c)
    zips = ([BUNDLE_ZIP] if BUNDLE_ZIP else []) + glob.glob("/kaggle/input/**/bde_bundle_core*.zip", recursive=True) + glob.glob("/content/**/bde_bundle_core*.zip", recursive=True) + glob.glob(str(Path.cwd() / "bde_bundle_core*.zip"))
    for z in zips:
        if z and Path(z).exists():
            print("unzipping", Path(z).name); zipfile.ZipFile(z).extractall(WORK)
            for c in (WORK / "bde-team7", WORK):
                if looks_ok(c): return c
    dest = WORK / "bde-team7"
    if not dest.exists(): subprocess.run(["git", "clone", "-q", "--depth", "1", GITHUB_URL, str(dest)])
    if looks_ok(dest): return dest
    raise SystemExit("Project files not found: upload bde_bundle_core.zip (Colab: Files panel; Kaggle: Add Data) and rerun.")

ROOT = find_root(); os.environ["BDE_ROOT"] = str(ROOT); os.chdir(ROOT)
print("platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local machine", "| project folder:", ROOT.name, "| cpus:", os.cpu_count())

platform: local machine | project folder: bde-team7 | cpus: 12


In [2]:
import importlib
miss = [p for m, p in {"xgboost": "xgboost", "pyarrow": "pyarrow", "sklearn": "scikit-learn", "scipy": "scipy"}.items() if importlib.util.find_spec(m) is None]
if miss: subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + miss)
import torch                                   # import torch BEFORE pandas (matters on Windows; harmless elsewhere)
import numpy as np, pandas as pd
sys.path.insert(0, str(ROOT / "modeling")); sys.path.insert(0, str(ROOT / "src"))
print("torch", torch.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

torch 2.14.0+cpu | pandas 3.0.2 | numpy 2.5.3


## Part 2 · Check the data
`processed/` is the committed output of the preprocessing pipeline (pandas reference, reproduced by the Spark and BigQuery versions). 60 of 84 columns are safe model inputs (class A), 13 are descriptive only (B), 11 are labels (C), see `docs/handoff_for_modeling.md`.

In [3]:
import data as D
b = D.build()
m = b["meta"]
print("model rows (eval-eligible):", len(m), "| inputs A / B:", len(b["names"]) - b["is_rain"].sum(), "/", len(b["names"]), "| sequence tensor:", b["seq"].shape)
display(pd.crosstab(m.split_chrono, m.season).loc[["train", "val", "test"]])
print("forbidden location/year columns among the inputs:", sorted({"lat", "lon", "state", "district", "year"} & set(b["names"])) or "none")

model rows (eval-eligible): 160580 | inputs A / B: 42 / 66 | sequence tensor: (160580, 13, 10)


season,post_monsoon_rabi,pre_monsoon,monsoon,post_monsoon_kharif
split_chrono,,,,
train,28364,22938,24990,28145
val,2668,2183,5298,5364
test,13298,6266,10470,10596


forbidden location/year columns among the inputs: none


## Part 3 · Code map
Every script of the modeling study with the first line of its docstring, and the order `run_pipeline.py --mode full` runs them.

In [4]:
import ast
files = sorted((ROOT / "modeling").glob("*.py"))
rows = []
for f in files:
    doc = (ast.get_docstring(ast.parse(f.read_text(encoding="utf-8"))) or "").strip().split("\n")[0]
    rows.append([f.name, len(f.read_text(encoding="utf-8").splitlines()), doc[:110]])
display(pd.DataFrame(rows, columns=["file in modeling/", "lines", "purpose (first docstring line)"]))

,file in modeling/,lines,purpose (first docstring line)
0,aggregate.py,267,Collect chronological/geographic results into ...
1,data.py,185,Model-ready arrays built ONLY from class-A col...
2,export_powerbi.py,39,Flat CSVs for Power BI / Tableau (results/powe...
3,gap_test.py,32,
4,interval_eval.py,77,H1: interval quality (80% intervals). Needs re...
5,lib.py,248,"Training, prediction and metric helpers shared..."
6,make_dashboard.py,228,"Interactive dashboard (single HTML file, Plotl..."
7,nets.py,216,"Deep models: LSTM [14], 1D-CNN [8],[15], ANN/M..."
8,robust_eval.py,53,H2: degradation under missing recent history. ...
9,robust_job.py,53,"Retrain one model (Model B, best validation co..."


In [5]:
import inspect, importlib.util
spec = importlib.util.spec_from_file_location("run_pipeline", ROOT / "modeling" / "run_pipeline.py"); rp = importlib.util.module_from_spec(spec); spec.loader.exec_module(rp)
print("Pipeline order (full mode):")
for name in ("step_chrono", "step_geo", "step_robust", "step_evaluate"):
    print(f"\n--- {name} ---"); print(inspect.getsource(getattr(rp, name)))

Pipeline order (full mode):

--- step_chrono ---
def step_chrono():
    LOGS.mkdir(parents=True, exist_ok=True)
    th = max(1, NCPU // 4)
    jobs = [(["run_chrono.py", m], {"BDE_THREADS": th}, LOGS / f"chrono_{m}.log") for m in TABULAR + DEEP]
    print(f"[chrono] {len(jobs)} models in parallel ({th} threads each), then TGT-M", flush=True)
    pool(jobs, 3)
    print("[chrono] TGT-M: tune 3 configs x 2 variants (parallel, 1 thread each), then seeds 1 and 2 for the best config", flush=True)
    pool([(["tgtm_job.py", v, str(c), "0"], {"BDE_THREADS": 1}, LOGS / f"tgtm_{v}_c{c}_s0.log") for v in "AB" for c in range(3)], max(1, min(6, NCPU)))
    import json
    best = {}
    for v in "AB":
        t = [json.loads((ROOT / "results" / "chrono" / "jobs" / f"TGT-M_{v}_c{c}_s0.json").read_text())["val_anom_MAE"] for c in range(3)]
        best[v] = t.index(min(t))
    pool([(["tgtm_job.py", v, str(best[v]), str(s)], {"BDE_THREADS": 1}, LOGS / f"tgtm_{v}_c{best[v]}_s{s}.log") for v in "AB" fo

In [6]:
import lib as Lb
print("Hyperparameter grids (3-4 configurations per model, chosen on VALIDATION only):")
for k, v in Lb.GRIDS.items():
    print(f"  {k:8s} {len(v)} configs, e.g. {v[0]}")

Hyperparameter grids (3-4 configurations per model, chosen on VALIDATION only):
  XGBoost  4 configs, e.g. {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 20, 'reg_lambda': 5}
  RF       4 configs, e.g. {'n_estimators': 150, 'max_depth': 12, 'min_samples_leaf': 20, 'max_features': 0.5}
  SVR      4 configs, e.g. {'nys': 400, 'gamma': 0.02, 'C': 0.5}
  ANN      3 configs, e.g. {'arch': {'hidden': 64, 'drop': 0.1}, 'lr': 0.002}
  CNN1D    3 configs, e.g. {'arch': {'hidden': 32, 'drop': 0.1}, 'lr': 0.002}
  LSTM     3 configs, e.g. {'arch': {'hidden': 32, 'drop': 0.1}, 'lr': 0.002}
  TGT-M    3 configs, e.g. {'arch': {'hidden': 32, 'drop': 0.2, 'heads': 2}, 'lr': 0.001, 'wd': 0.001}


## Part 4 · Run the models
* **view** does nothing here.
* **quick** retrains XGBoost and ANN (one seed). Expected: XGBoost A 1.264 / B 1.120 m; ANN seed 0 A 1.300 / B 1.142 m test depth MAE (the report's 3-seed ANN means are 1.270 / 1.125).
* **full** runs `modeling/run_pipeline.py --mode full --steps STEPS`; logs go to `results/logs/` and the run **overwrites** `results/`.

In [7]:
t0 = time.time()
if MODE == "quick":
    r = subprocess.run([sys.executable, "-u", "modeling/run_pipeline.py", "--mode", "quick"], cwd=ROOT, capture_output=True, text=True)
    print(r.stdout[-3500:]); print(r.stderr[-1200:])
    print("quick run finished in %.0f s (exit code %d)" % (time.time() - t0, r.returncode))
elif MODE == "full":
    log = open(ROOT / "full_pipeline.log", "w")
    p = subprocess.Popen([sys.executable, "-u", "modeling/run_pipeline.py", "--mode", "full", "--steps", STEPS], cwd=ROOT, stdout=log, stderr=subprocess.STDOUT)
    print("full pipeline started in the background (pid %d); monitor it with the next cell." % p.pid)
else:
    print("MODE = view: nothing to train. The stored results below come from the full study.")

[quick] XGBoost + ANN, seed 0 only, output in results_quick/ (results/ is untouched)
  run_chrono.py XGBoost  [225s] ok
  run_chrono.py ANN  [238s] ok

model  variant  test depth MAE  anomaly R2
XGBoost  A   1.264          0.186
XGBoost  B   1.120          0.278
ANN      A   1.300          0.156
ANN      B   1.142          0.277

Expected: XGBoost A 1.264 / B 1.120; ANN seed 0 A 1.300 / B 1.142 (the report shows 3-seed means: ANN 1.270 / 1.125). Differences of about 0.01 m are normal.


quick run finished in 238 s (exit code 0)


In [8]:
if MODE == "full":
    print(subprocess.run(["tail", "-n", "15", str(ROOT / "full_pipeline.log")], capture_output=True, text=True).stdout)
    for d in ("chrono", "geo", "robust"):
        print(d, len(list((ROOT / "results" / d).glob("*"))), "files")
else:
    print("(monitor cell: only used in full mode)")

(monitor cell: only used in full mode)


## Part 5 · Result tables
Every number in the report, paper and slides comes from `results/tables/`.

In [9]:
T = ROOT / "results" / "tables"
s = pd.read_csv(T / "chrono_summary.csv")
print("Main comparison, test split 2018-2022 (mean and sd over 3 seeds; A = no rainfall, B = with rainfall):")
display(s[["model", "variant", "depth_MAE_mean", "depth_MAE_std", "anom_MAE_mean", "anom_R2_mean", "macro_F1_mean", "AUC_stressed_mean"]].round(4))

Main comparison, test split 2018-2022 (mean and sd over 3 seeds; A = no rainfall, B = with rainfall):


,model,variant,depth_MAE_mean,depth_MAE_std,anom_MAE_mean,anom_R2_mean,macro_F1_mean,AUC_stressed_mean
0,Prev. round,A,2.0502,NaN,1.4969,-1.7310,0.3222,0.6801
1,Prev. round,B,2.0502,NaN,1.4969,-1.7310,0.3222,0.6801
2,Same season LY,A,1.5070,NaN,0.9809,-0.1745,0.3454,0.6783
3,Same season LY,B,1.5070,NaN,0.9809,-0.1745,0.3454,0.6783
4,Seasonal baseline,A,1.5484,NaN,0.9674,-0.0173,0.2284,0.5000
5,Seasonal baseline,B,1.5484,NaN,0.9674,-0.0173,0.2284,0.5000
6,SVR,A,1.4589,0.0040,0.9453,0.0351,0.3368,0.7360
7,SVR,B,1.1799,0.0067,0.7904,0.2498,0.3125,0.7809
8,RF,A,1.2505,0.0007,0.8334,0.1947,0.3128,0.7603
9,RF,B,1.1501,0.0003,0.7767,0.2634,0.3132,0.7809


In [10]:
for name, note in [("bootstrap.csv", "Rainfall effect (B minus A) and TGT-M versus the others, 95% bootstrap CI over wells"),
                   ("geo_summary.csv", "Unseen-state results (g5 = grouped 5-fold, loso = leave-one-state-out)"),
                   ("interval_summary.csv", "Pre-registered H1: 80% intervals"), ("interval_comparison.csv", "H1: TGT-M interval score minus the others (negative = TGT-M better)"),
                   ("robust_summary.csv", "Pre-registered H2: anomaly MAE versus share of recent readings removed"), ("robust_degradation.csv", "H2: degradation and differences with CI"),
                   ("generalization_gap.csv", "Post hoc: unseen minus seen error")]:
    f = T / name
    if f.exists():
        print("\n" + note); display(pd.read_csv(f).round(4))


Rainfall effect (B minus A) and TGT-M versus the others, 95% bootstrap CI over wells


,comparison,metric,diff,ci_lo,ci_hi
0,SVR: B - A (rainfall effect),depth_MAE,-0.2790,-0.2923,-0.2661
1,SVR: B - A (rainfall effect),anom_MAE,-0.1549,-0.1594,-0.1501
2,RF: B - A (rainfall effect),depth_MAE,-0.1004,-0.1084,-0.0925
3,RF: B - A (rainfall effect),anom_MAE,-0.0567,-0.0602,-0.0531
4,ANN: B - A (rainfall effect),depth_MAE,-0.1455,-0.1540,-0.1369
5,ANN: B - A (rainfall effect),anom_MAE,-0.0779,-0.0817,-0.0739
6,XGBoost: B - A (rainfall effect),depth_MAE,-0.1424,-0.1518,-0.1329
7,XGBoost: B - A (rainfall effect),anom_MAE,-0.0768,-0.0808,-0.0730
8,CNN1D: B - A (rainfall effect),depth_MAE,-0.1739,-0.1836,-0.1642
9,CNN1D: B - A (rainfall effect),anom_MAE,-0.0932,-0.0975,-0.0891



Unseen-state results (g5 = grouped 5-fold, loso = leave-one-state-out)


,model,scheme,variant,n,depth_MAE,depth_RMSE,depth_R2,anom_MAE,anom_RMSE,anom_R2,...,P_Watch,R_Watch,F1_Watch,P_Moderate Stress,R_Moderate Stress,F1_Moderate Stress,P_High Stress,R_High Stress,F1_High Stress,AUC_stressed
0,ANN,g5,A,40630,1.2592,2.0184,0.8241,0.8395,1.1297,0.1763,...,0.1940,0.1506,0.1695,0.1956,0.0965,0.1293,0.5102,0.0791,0.1370,0.7560
1,ANN,g5,B,40630,1.1389,1.8417,0.8535,0.7709,1.0649,0.2681,...,0.2271,0.1124,0.1504,0.2535,0.0777,0.1190,0.6346,0.0696,0.1255,0.7850
2,CNN1D,g5,A,40630,1.2792,2.0311,0.8218,0.8455,1.1343,0.1695,...,0.2051,0.1684,0.1850,0.1896,0.0971,0.1285,0.4408,0.0844,0.1417,0.7629
3,CNN1D,g5,B,40630,1.1235,1.8298,0.8554,0.7601,1.0560,0.2803,...,0.2089,0.1110,0.1450,0.2281,0.0808,0.1193,0.5479,0.0965,0.1641,0.7946
4,LSTM,g5,A,40630,1.2586,2.0071,0.8260,0.8341,1.1243,0.1841,...,0.1941,0.1568,0.1734,0.1795,0.0923,0.1219,0.4624,0.0844,0.1427,0.7642
5,LSTM,g5,B,40630,1.1333,1.8326,0.8550,0.7634,1.0597,0.2753,...,0.2114,0.1234,0.1558,0.2284,0.0868,0.1258,0.5816,0.1034,0.1755,0.7906
6,RF,g5,A,40630,1.2668,2.0056,0.8263,0.8402,1.1237,0.1849,...,0.1860,0.1224,0.1476,0.1952,0.0589,0.0905,0.5299,0.0374,0.0700,0.7552
7,RF,g5,B,40630,1.1618,1.8695,0.8491,0.7826,1.0727,0.2572,...,0.2136,0.0990,0.1353,0.2322,0.0595,0.0947,0.5490,0.0443,0.0820,0.7774
8,SVR,g5,A,40630,1.4188,2.2143,0.7882,0.9200,1.2012,0.0687,...,0.1624,0.1901,0.1751,0.1866,0.0929,0.1240,0.4542,0.0707,0.1223,0.7341
9,SVR,g5,B,40630,1.1853,1.9336,0.8385,0.7894,1.0799,0.2473,...,0.2022,0.1114,0.1436,0.2292,0.0619,0.0975,0.6241,0.0464,0.0864,0.7784



Pre-registered H1: 80% intervals


...

,model,p,mean,std,count
0,ANN,0.0,0.7597,0.0069,3
1,ANN,0.2,0.7875,0.0080,3
2,ANN,0.4,0.8169,0.0089,3
3,ANN,0.6,0.8471,0.0106,3
4,CNN1D,0.0,0.7556,0.0037,3
5,CNN1D,0.2,0.7830,0.0017,3
6,CNN1D,0.4,0.8122,0.0014,3
7,CNN1D,0.6,0.8445,0.0042,3
8,LSTM,0.0,0.7475,0.0020,3
9,LSTM,0.2,0.7748,0.0014,3



H2: degradation and differences with CI


,comparison,what,estimate,ci_lo,ci_hi
0,SVR,degradation D (p=0.6 minus p=0),0.0893,0.0845,0.0938
1,RF,degradation D (p=0.6 minus p=0),0.0595,0.0550,0.0640
2,ANN,degradation D (p=0.6 minus p=0),0.0874,0.0827,0.0920
3,XGBoost,degradation D (p=0.6 minus p=0),0.0897,0.0850,0.0942
4,CNN1D,degradation D (p=0.6 minus p=0),0.0889,0.0841,0.0938
5,LSTM,degradation D (p=0.6 minus p=0),0.0911,0.0863,0.0961
6,TGT-M,degradation D (p=0.6 minus p=0),0.0823,0.0780,0.0865
7,TGT-M minus SVR,difference of degradations,-0.0070,-0.0098,-0.0040
8,TGT-M minus SVR,difference of MAE at p=0.6,-0.0347,-0.0379,-0.0312
9,TGT-M minus RF,difference of degradations,0.0227,0.0193,0.0260



Post hoc: unseen minus seen error


,comparison,what,estimate,ci_lo,ci_hi
0,SVR,own gap,-0.0043,-0.0065,-0.0021
1,RF,own gap,0.0058,0.0045,0.0071
2,ANN,own gap,0.0036,0.0015,0.0057
3,XGBoost,own gap,0.0038,0.0028,0.0049
4,CNN1D,own gap,0.0160,0.0141,0.0181
5,LSTM,own gap,0.0170,0.0147,0.0191
6,TGT-M,own gap,-0.0040,-0.0059,-0.0020
7,TGT-M minus SVR,gap diff,0.0003,-0.0021,0.0027
8,TGT-M minus RF,gap diff,-0.0098,-0.0118,-0.0078
9,TGT-M minus ANN,gap diff,-0.0076,-0.0095,-0.0055


## Appendix
**Reading the numbers.** *Depth MAE* = mean absolute error of the predicted groundwater depth in metres (lower is better). *Anomaly* = (reading − the well's past same-season mean) / its SD, capped at ±5. *Macro-F1* = mean F1 over the 4 stress classes. *Interval score* = interval width + 10 × the amount by which the truth falls outside the 80% interval (lower is better). TGT-M ranks 5th of 7 learned models on accuracy; rainfall improves every model; H1 and H2a (pre-registered) are supported, H2b is not.

**Troubleshooting.** *Project files not found* → upload `bde_bundle_core.zip` and re-run Part 1. *Out of memory in full mode* → run one `STEPS` value at a time. Neural networks are not bit-for-bit reproducible across machines: differences of about 0.01 m are normal.
